In [ ]:
# =========================
# THESIS 4-SCENARIO PIPELINE
# Colab-ready | Google Drive output
# Dataset: All.Engineering.Papers_5000.csv
# Columns: doc_id, year, field, text_fa_raw, text_en_raw, pseudo_label
# =========================

!pip install -q pandas numpy scikit-learn matplotlib seaborn openpyxl xlsxwriter sentence-transformers transformers torch hazm

from google.colab import drive
drive.mount('/content/drive')

from __future__ import annotations
import os
import re
import json
import time
import math
import zipfile
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, PCA
from sklearn.preprocessing import normalize, MinMaxScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    silhouette_score,
    normalized_mutual_info_score,
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

from sentence_transformers import SentenceTransformer

# =========================
# Persian/English text-cleaning resources (added)
# Uses hazm if available; otherwise falls back to a built-in stopword list.
# =========================
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
EN_STOP = set(ENGLISH_STOP_WORDS)

# Built-in fallback Persian stopword list (used only if hazm is unavailable)
_FA_STOP_FALLBACK = set("""
و در به از که این را با است برای آن یک تا های می ها هم خود نیز شده بر اما یا اگر
بود کرد باید چه چون هر همه دو شد دارد شود مورد بین حتی چند طور دیگر پس کنند نمی
بی روی همین بسیار بدون پیش وی ای آنها ایم اند کنیم کند داده مانند طی توسط همچنین
گرفته داشته باشد نسبت زیرا چنین آیا کدام کسی چرا کجا اینکه آنکه گردد گفت داشت
میشود میباشد بهعنوان نتایج روش مدل استفاده بهبود دهد دارند کرده شدن آنان خواهد
""".split())

_HAS_HAZM = False
try:
    from hazm import Normalizer as _HazmNormalizer, Lemmatizer as _HazmLemmatizer, \
                     word_tokenize as _hazm_word_tokenize, stopwords_list as _hazm_stopwords_list
    _fa_normalizer = _HazmNormalizer()
    _fa_lemmatizer = _HazmLemmatizer()
    FA_STOP = set(_hazm_stopwords_list())
    from functools import lru_cache
    @lru_cache(maxsize=300000)
    def _lemma_cached(tok):
        # lemmatize each unique token only once (verbs come back as "نوشت#نویس" -> keep first)
        return _fa_lemmatizer.lemmatize(tok).split("#")[0]
    _HAS_HAZM = True
    print("[preprocess] hazm loaded — using Normalizer + Lemmatizer + official stopwords.")
except Exception as _e:
    FA_STOP = _FA_STOP_FALLBACK
    print("[preprocess] hazm NOT available — using regex normalizer + fallback stopword list.")

# Min token length to keep (drops most prepositions / single letters)
MIN_TOKEN_LEN = 3

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")


# =========================
# CONFIG
# =========================
@dataclass
class Config:
    input_csv: str = "/content/drive/MyDrive/deep-embedd/All.Engineering.Papers_5000.csv"
    out_root: str = "/content/drive/MyDrive/deep-embedd/thesis_outputs"

    # --- real dataset columns (All.Engineering.Papers.csv) ---
    doc_id_col: str = "a"          # paper number
    title_fa_col: str = "b"        # Persian title
    title_en_col: str = "c"        # English title
    fa_col: str = "d"              # Persian full text
    en_col: str = "e"              # English full text
    kw_fa_col: str = "f"           # Persian keywords
    kw_en_col: str = "g"           # English top words
    # pseudo label is OPTIONAL. If this column is absent -> fully unsupervised.
    pseudo_col: str = "pseudo_label"
    year_col: str = "year"         # optional
    field_col: str = "field"       # optional

    # --- multilabel assignment (hybrid: up to TOP_K, only above threshold) ---
    multilabel_top_k: int = 3
    multilabel_threshold: float = 0.15

    # --- keyword boosting ---
    # author keywords (cols f,g) carry concentrated topical signal, so we repeat them
    # `keyword_weight` times when building the input text. 0 = ignore keywords.
    use_keywords: bool = True
    keyword_weight: int = 3

    random_state: int = 42

    min_df: int = 5
    max_df_ratio: float = 0.85
    max_features: int = 30000

    lda_k_grid: Tuple[int, ...] = (10, 15, 20, 25, 30)
    lda_max_iter: int = 20
    lda_learning_method: str = "batch"
    top_words_per_topic: int = 12

    topic_merge_cos_thr: float = 0.92
    embed_merge_cos_thr: float = 0.95

    rho_grid: Tuple[float, ...] = (0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85, 0.90)
    # complement coding (standard Fuzzy ART preprocessing) prevents category proliferation.
    use_complement_coding: bool = True
    alpha: float = 1e-6
    beta: float = 1.0

    lambda_fa: float = 0.5
    en_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    fa_model_name: str = "myrkur/sentence-transformer-parsbert-fa"   # ParsBERT, Persian only
    batch_size_embed: int = 32

    fig_dpi: int = 300
    top_clusters_to_report: int = 10

CFG = Config()

SCENARIOS = {
    "S1_LDA_ENTM_SSFuzzyART": {"feature_type": "lda",  "use_entm": True},
    "S2_LDA_SSFuzzyART":      {"feature_type": "lda",  "use_entm": False},
    "S3_BERT_ENTM_SSFuzzyART":{"feature_type": "bert", "use_entm": True},
    "S4_BERT_SSFuzzyART":     {"feature_type": "bert", "use_entm": False},
}


# =========================
# HELPERS
# =========================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def save_json(obj, path):
    ensure_dir(Path(path).parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def normalize_fa(text):
    text = safe_text(text)
    text = text.replace("&nbsp;", " ").replace("&amp;", " ")   # strip HTML entities
    if _HAS_HAZM:
        # hazm: standard normalization (chars, ZWNJ, diacritics)
        text = _fa_normalizer.normalize(text)
        # drop digits / latin / punctuation, keep Persian letters and spaces
        text = re.sub(r"[\u064B-\u0652\u0640]", "", text)
        text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        tokens = _hazm_word_tokenize(text)
        cleaned = []
        for t in tokens:
            lem = _lemma_cached(t)            # cached: each unique token lemmatized only once
            if lem and lem not in FA_STOP and len(lem) >= MIN_TOKEN_LEN:
                cleaned.append(lem)
        return " ".join(cleaned)
    # ---- fallback (no hazm) ----
    text = re.sub(r"[\u064B-\u0652\u0640]", "", text)            # diacritics + tatweel
    for a, b in {"ي":"ی","ك":"ک","أ":"ا","إ":"ا","آ":"ا","ٱ":"ا",
                 "ة":"ه","ؤ":"و","ئ":"ی"}.items():
        text = text.replace(a, b)                                  # unify Arabic chars
    text = text.replace("\u200c", " ")                             # ZWNJ
    text = re.sub(r"[0-9\u06F0-\u06F9]+", " ", text)              # digits
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)              # keep Persian only
    text = re.sub(r"\s+", " ", text).strip()
    toks = [t for t in text.split() if t not in FA_STOP and len(t) >= MIN_TOKEN_LEN]
    return " ".join(toks)

def normalize_en(text):
    text = safe_text(text).lower()
    text = text.replace("&nbsp;", " ").replace("&amp;", " ")   # strip HTML entities
    text = re.sub(r"[^a-z\s]", " ", text)                          # drop digits + symbols
    text = re.sub(r"\s+", " ", text).strip()
    toks = [t for t in text.split() if t not in EN_STOP and len(t) >= MIN_TOKEN_LEN]
    return " ".join(toks)

def minmax_01(X):
    scaler = MinMaxScaler()
    return scaler.fit_transform(X)

def entropy_row(p):
    p = np.asarray(p, dtype=float)
    s = p.sum()
    if s <= 0:
        return 0.0
    p = p / s
    p = np.clip(p, 1e-12, 1.0)
    return float(-(p * np.log(p)).sum())

def topic_diversity(topic_words_lists):
    all_words = []
    for words in topic_words_lists:
        all_words.extend(words)
    if len(all_words) == 0:
        return np.nan
    return len(set(all_words)) / len(all_words)

def zip_all_outputs(src_dir: Path, zip_path: Path):
    ensure_dir(zip_path.parent)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in src_dir.rglob("*"):
            if p.is_file() and p != zip_path:
                zf.write(p, arcname=p.relative_to(src_dir))

def plot_bar(df, xcol, ycol, title, out_path, rotate=25):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df[xcol].astype(str), df[ycol], alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    fig.savefig(out_path, dpi=CFG.fig_dpi)
    plt.close(fig)


# =========================
# DATA LOADING & PREPROCESS
# =========================
def load_dataset():
    # The dataset file has NO header row; columns are positional.
    # Expected order: a=paper_no, b=title_fa, c=title_en, d=text_fa, e=text_en,
    #                 f=keywords_fa, g=topwords_en, then optional extra columns (ISSN, etc.)
    raw = pd.read_csv(CFG.input_csv, encoding="utf-8-sig", header=None, dtype=str)

    base_names = [CFG.doc_id_col, CFG.title_fa_col, CFG.title_en_col,
                  CFG.fa_col, CFG.en_col, CFG.kw_fa_col, CFG.kw_en_col]
    ncols = raw.shape[1]
    if ncols < len(base_names):
        raise ValueError(f"Expected at least {len(base_names)} columns, found {ncols}.")
    # name the first 7 by role; keep any extras as extra_0, extra_1, ...
    names = base_names + [f"extra_{i}" for i in range(ncols - len(base_names))]
    raw.columns = names
    df = raw

    # clean obvious junk: literal "NULL" -> empty
    for c in [CFG.fa_col, CFG.en_col, CFG.kw_fa_col, CFG.kw_en_col,
              CFG.title_fa_col, CFG.title_en_col]:
        df[c] = df[c].replace("NULL", np.nan)

    required = [CFG.fa_col, CFG.en_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required text columns: {missing}. Found: {list(df.columns)}")

    # doc id
    if CFG.doc_id_col not in df.columns:
        df.insert(0, CFG.doc_id_col, np.arange(1, len(df) + 1))
    # fall back to a sequential id only where paper number is missing
    df[CFG.doc_id_col] = df[CFG.doc_id_col].fillna(
        pd.Series(np.arange(1, len(df) + 1), index=df.index).astype(str)
    )

    # optional columns -> create empty placeholders so the rest of the pipeline never crashes
    for opt in [CFG.year_col, CFG.field_col, CFG.title_fa_col, CFG.title_en_col,
                CFG.kw_fa_col, CFG.kw_en_col]:
        if opt not in df.columns:
            df[opt] = ""

    # PSEUDO LABEL is optional:
    #   - if the column exists and has any non-empty value -> SEMI-SUPERVISED mode
    #   - otherwise -> a NaN column is created and the run is FULLY UNSUPERVISED
    if CFG.pseudo_col in df.columns:
        df[CFG.pseudo_col] = df[CFG.pseudo_col].where(df[CFG.pseudo_col].notna(), np.nan)
        n_lab = int(df[CFG.pseudo_col].notna().sum())
        print(f"[data] pseudo_label found -> SEMI-SUPERVISED ({n_lab} labeled rows).")
    else:
        df[CFG.pseudo_col] = np.nan
        print("[data] no pseudo_label column -> FULLY UNSUPERVISED.")

    return df

def has_any_labels(series) -> bool:
    """True if at least 2 distinct non-empty labels exist (needed to seed/evaluate)."""
    try:
        vals = series.dropna()
        vals = vals[vals.astype(str).str.strip() != ""]
        return vals.nunique() >= 2
    except Exception:
        return False

def preprocess_dataset(df: pd.DataFrame, out_dir: Path, force: bool = False):
    ensure_dir(out_dir)

    # ---- cache: if preprocessing already ran, just load it (skip the slow step) ----
    cache_path = out_dir / "preprocessed_dataset.csv"
    if cache_path.exists() and not force:
        print(f"[preprocess] cache found -> loading {cache_path} (set force=True to rebuild)")
        d = pd.read_csv(cache_path, encoding="utf-8-sig")
        for col in ["text_fa_clean", "text_en_clean", "text_joined"]:
            if col in d.columns:
                d[col] = d[col].fillna("").astype(str)
        d["has_fa"] = d["text_fa_clean"].str.len() > 0
        d["has_en"] = d["text_en_clean"].str.len() > 0
        return d

    print("[preprocess] building preprocessed dataset (this runs once)...")
    d = df.copy()
    d["text_fa_clean"] = d[CFG.fa_col].fillna("").astype(str).apply(normalize_fa)
    d["text_en_clean"] = d[CFG.en_col].fillna("").astype(str).apply(normalize_en)

    # --- keywords (cols f,g): cleaned, then repeated `keyword_weight` times to boost them ---
    # Missing keywords -> empty string, so papers without keywords simply fall back to text only.
    if CFG.use_keywords and CFG.keyword_weight > 0:
        kw_fa = d.get(CFG.kw_fa_col)
        kw_en = d.get(CFG.kw_en_col)
        d["kw_fa_clean"] = (kw_fa.fillna("").astype(str).apply(normalize_fa)
                            if kw_fa is not None else "")
        d["kw_en_clean"] = (kw_en.fillna("").astype(str).apply(normalize_en)
                            if kw_en is not None else "")
        kw_boost = (
            (d["kw_fa_clean"].fillna("") + " " + d["kw_en_clean"].fillna("")).str.strip()
            + " "
        ) * CFG.keyword_weight
    else:
        d["kw_fa_clean"] = ""
        d["kw_en_clean"] = ""
        kw_boost = ""

    d["text_joined"] = (
        d["text_fa_clean"].fillna("") + " " +
        d["text_en_clean"].fillna("") + " " +
        kw_boost
    ).str.strip()
    d["text_joined"] = d["text_joined"].str.replace(r"\s+", " ", regex=True).str.strip()

    d["has_fa"] = d["text_fa_clean"].str.len() > 0
    d["has_en"] = d["text_en_clean"].str.len() > 0
    d["has_kw"] = (d["kw_fa_clean"].str.len() + d["kw_en_clean"].str.len()) > 0

    d.to_csv(out_dir / "preprocessed_dataset.csv", index=False, encoding="utf-8-sig")

    stats = {
        "n_docs": int(len(d)),
        "n_has_fa": int(d["has_fa"].sum()),
        "n_has_en": int(d["has_en"].sum()),
        "n_has_both": int((d["has_fa"] & d["has_en"]).sum()),
        "pseudo_label_nunique": int(d[CFG.pseudo_col].dropna().nunique()) if CFG.pseudo_col in d.columns else 0,
        "mode": "semi_supervised" if has_any_labels(d[CFG.pseudo_col]) else "unsupervised"
    }
    save_json(stats, out_dir / "preprocess_stats.json")
    return d


# =========================
# LDA FEATURES
# =========================
def build_lda_features(df1: pd.DataFrame, out_dir: Path):
    ensure_dir(out_dir)

    vectorizer = CountVectorizer(
        min_df=CFG.min_df,
        max_df=CFG.max_df_ratio,
        max_features=CFG.max_features,
        stop_words=list(FA_STOP | EN_STOP),     # second safety layer
        token_pattern=r"(?u)\b\w{3,}\b"        # tokens with >= 3 chars
    )
    X_bow = vectorizer.fit_transform(df1["text_joined"].fillna(""))

    k_records = []
    best_score = None
    best_model = None
    best_k = None

    for k in CFG.lda_k_grid:
        lda = LatentDirichletAllocation(
            n_components=k,
            random_state=CFG.random_state,
            max_iter=CFG.lda_max_iter,
            learning_method=CFG.lda_learning_method
        )
        doc_topic = lda.fit_transform(X_bow)
        perp = lda.perplexity(X_bow)
        score = lda.score(X_bow)
        rec = {"k": k, "perplexity": float(perp), "log_likelihood": float(score)}
        k_records.append(rec)

        if best_score is None or score > best_score:
            best_score = score
            best_model = lda
            best_k = k

    k_df = pd.DataFrame(k_records)
    k_df.to_csv(out_dir / "lda_k_search.csv", index=False, encoding="utf-8-sig")

    lda_final = best_model
    doc_topic = lda_final.transform(X_bow)
    doc_topic = normalize(doc_topic, norm="l1")

    vocab = np.array(vectorizer.get_feature_names_out())
    topic_word = lda_final.components_ / lda_final.components_.sum(axis=1, keepdims=True)

    topics_rows = []
    topic_word_lists = []
    for t in range(topic_word.shape[0]):
        idx = np.argsort(topic_word[t])[::-1][:CFG.top_words_per_topic]
        words = vocab[idx].tolist()
        topic_word_lists.append(words)
        topics_rows.append({
            "topic_id": t,
            "top_words": " | ".join(words),
            "topic_entropy": entropy_row(topic_word[t])
        })
    topics_df = pd.DataFrame(topics_rows)
    topics_df.to_csv(out_dir / "lda_topics_top_words.csv", index=False, encoding="utf-8-sig")

    pd.DataFrame(doc_topic).to_csv(out_dir / "lda_doc_topic.csv", index=False, encoding="utf-8-sig")

    meta = {
        "feature_type": "lda",
        "best_k": int(best_k),
        "topic_diversity": float(topic_diversity(topic_word_lists)),
        "vocab_size": int(len(vocab))
    }
    save_json(meta, out_dir / "lda_meta.json")

    return {
        "X_features": doc_topic,
        "lda_model": lda_final,
        "vectorizer": vectorizer,
        "topic_word": topic_word,
        "topics_df": topics_df,
        "topic_word_lists": topic_word_lists,
        "meta": meta
    }


# =========================
# BERT / MULTILINGUAL FEATURES
# =========================
def build_bert_features(df1: pd.DataFrame, out_dir: Path):
    ensure_dir(out_dir)

    print("Loading embedding models...")
    model_en = SentenceTransformer(CFG.en_model_name)
    model_fa = SentenceTransformer(CFG.fa_model_name)

    fa_texts = df1["text_fa_clean"].fillna("").astype(str).tolist()
    en_texts = df1["text_en_clean"].fillna("").astype(str).tolist()

    print("Encoding Persian texts...")
    E_fa = model_fa.encode(
        fa_texts,
        batch_size=CFG.batch_size_embed,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    print("Encoding English texts...")
    E_en = model_en.encode(
        en_texts,
        batch_size=CFG.batch_size_embed,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    has_fa = df1["has_fa"].values.astype(bool)
    has_en = df1["has_en"].values.astype(bool)

    # ParsBERT (Persian) and the English model usually have DIFFERENT dimensions
    # (e.g. 768 vs 384), so they cannot be averaged. We CONCATENATE the two parts.
    # Each part is weighted; a missing language for a doc contributes zeros to its block.
    d_fa = E_fa.shape[1]
    d_en = E_en.shape[1]
    w_fa = float(CFG.lambda_fa)
    w_en = 1.0 - w_fa
    E = np.zeros((len(df1), d_fa + d_en), dtype=np.float32)
    for i in range(len(df1)):
        if has_fa[i]:
            E[i, :d_fa] = w_fa * E_fa[i]
        if has_en[i]:
            E[i, d_fa:] = w_en * E_en[i]
        # if a doc has neither, its row stays all zeros (handled later)

    # z-score per dimension removes the ~0.5 offset that destroys contrast,
    # then L2-normalize so cosine similarity is meaningful for dense embeddings.
    from sklearn.preprocessing import StandardScaler
    E = StandardScaler().fit_transform(E)
    X_embed = normalize(E, norm="l2")

    pd.DataFrame(X_embed).to_csv(out_dir / "bert_parsbert_embeddings.csv", index=False, encoding="utf-8-sig")

    meta = {
        "feature_type": "bert",
        "en_model": CFG.en_model_name,
        "fa_model": CFG.fa_model_name,
        "embed_dim": int(X_embed.shape[1]),
        "lambda_fa": float(CFG.lambda_fa)
    }
    save_json(meta, out_dir / "bert_meta.json")

    return {
        "X_features": X_embed,
        "meta": meta
    }


# =========================
# ENTM-LIKE STABILIZATION
# =========================
def merge_redundant_dimensions_by_similarity(X, threshold=0.95):
    X = np.asarray(X, dtype=float)
    XT = X.T
    sim = cosine_similarity(XT)

    n = sim.shape[0]
    visited = np.zeros(n, dtype=bool)
    groups = []

    for i in range(n):
        if visited[i]:
            continue
        grp = [i]
        visited[i] = True
        for j in range(i + 1, n):
            if not visited[j] and sim[i, j] >= threshold:
                grp.append(j)
                visited[j] = True
        groups.append(grp)

    X_new = np.zeros((X.shape[0], len(groups)), dtype=float)
    for g_idx, grp in enumerate(groups):
        X_new[:, g_idx] = X[:, grp].mean(axis=1)

    X_new = normalize(np.clip(X_new, 1e-12, None), norm="l1" if X_new.sum() > 0 else "l2")
    return X_new, groups, sim

def run_entm_general(X0, feature_type, out_dir: Path):
    ensure_dir(out_dir)

    thr = CFG.topic_merge_cos_thr if feature_type == "lda" else CFG.embed_merge_cos_thr
    X1, groups, sim = merge_redundant_dimensions_by_similarity(X0, threshold=thr)

    pd.DataFrame(X1).to_csv(out_dir / "entm_stabilized_features.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame({
        "group_id": list(range(len(groups))),
        "merged_dims": ["|".join(map(str, g)) for g in groups],
        "group_size": [len(g) for g in groups]
    }).to_csv(out_dir / "entm_groups.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(8, 6))
    sns.heatmap(sim, cmap="viridis")
    plt.title(f"Similarity matrix before ENTM-like merge ({feature_type})")
    plt.tight_layout()
    plt.savefig(out_dir / "entm_similarity_heatmap.png", dpi=CFG.fig_dpi)
    plt.close()

    meta = {
        "entm_applied": True,
        "feature_type": feature_type,
        "merge_threshold": float(thr),
        "dim_before": int(X0.shape[1]),
        "dim_after": int(X1.shape[1]),
        "n_groups": int(len(groups))
    }
    save_json(meta, out_dir / "entm_meta.json")
    return X1, meta


# =========================
# SIMPLE SSFuzzyART-LIKE CLUSTERER
# =========================
class SSFuzzyART:
    def __init__(self, rho=0.85, alpha=1e-6, beta=1.0, metric="fuzzy"):
        self.rho = rho
        self.alpha = alpha
        self.beta = beta
        self.metric = metric            # "fuzzy" for LDA (sparse), "cosine" for BERT (dense)
        self.prototypes = None
        self.counts = None
        self.seed_label_of_proto = []   # maps prototype index -> seed label (or None)

    def _choice(self, x, w):
        if self.metric == "cosine":
            return self._cosine(x, w)
        return np.sum(np.minimum(x, w)) / (self.alpha + np.sum(w))

    def _match(self, x, w):
        if self.metric == "cosine":
            # cosine similarity, mapped from [-1,1] to [0,1] so rho in [0,1] is meaningful
            return 0.5 * (self._cosine(x, w) + 1.0)
        denom = max(np.sum(x), 1e-12)
        return np.sum(np.minimum(x, w)) / denom

    @staticmethod
    def _cosine(x, w):
        nx = np.linalg.norm(x); nw = np.linalg.norm(w)
        if nx < 1e-12 or nw < 1e-12:
            return 0.0
        return float(np.dot(x, w) / (nx * nw))

    def _update_proto(self, w, x):
        if self.metric == "cosine":
            new = w + x                       # running sum, then renormalize (centroid on sphere)
            n = np.linalg.norm(new)
            return new / n if n > 1e-12 else new
        return self.beta * np.minimum(x, w) + (1 - self.beta) * w

    def fit_predict(self, X, seed_labels=None):
        """
        X           : feature matrix (n_docs x n_features), values in [0,1].
        seed_labels : optional array (len n_docs). Non-null entries are KNOWN labels
                      used to initialize clusters (SEMI-SUPERVISED). Null/None -> ignored.
                      If seed_labels is None or all-null -> behaves FULLY UNSUPERVISED.
        Returns: labels (hard, argmax) and M (soft membership matrix).
        """
        X = np.asarray(X, dtype=float)

        # --- Complement coding (standard Fuzzy ART): A = [x, 1-x]. ---
        # Keeps every vector's L1 norm constant (= n_features), which is what
        # prevents category proliferation (Carpenter, Grossberg & Rosen, 1991).
        if self.metric == "fuzzy" and getattr(CFG, "use_complement_coding", True):
            Xc = np.clip(X, 0.0, 1.0)
            X = np.concatenate([Xc, 1.0 - Xc], axis=1)

        n = X.shape[0]
        labels = np.full(n, -1, dtype=int)
        memberships = []

        prototypes = []
        counts = []
        self.seed_label_of_proto = []

        # ---- SEMI-SUPERVISED seeding: one prototype per known class ----
        # built from the mean of all docs sharing the same seed label.
        seed_map = {}   # seed label -> prototype index
        if seed_labels is not None:
            seed_labels = np.asarray(seed_labels, dtype=object)
            uniq = [s for s in pd.unique(seed_labels)
                    if (s is not None) and (not (isinstance(s, float) and np.isnan(s)))
                    and str(s).strip() != ""]
            for s in uniq:
                mask = np.array([str(v) == str(s) for v in seed_labels])
                if mask.sum() == 0:
                    continue
                proto = X[mask].mean(axis=0)            # class centroid as initial prototype
                prototypes.append(proto.copy())
                counts.append(int(mask.sum()))
                self.seed_label_of_proto.append(s)
                seed_map[str(s)] = len(prototypes) - 1

        # ---- main online ART loop ----
        for i, x in enumerate(X):
            # if this doc is a labeled seed, force it into its class prototype
            forced = None
            if seed_labels is not None:
                sv = seed_labels[i]
                if (sv is not None) and (not (isinstance(sv, float) and np.isnan(sv))) and str(sv).strip() != "":
                    forced = seed_map.get(str(sv))

            if len(prototypes) == 0:
                prototypes.append(x.copy()); counts.append(1)
                self.seed_label_of_proto.append(None)
                labels[i] = 0; memberships.append([1.0]); continue

            choices = np.array([self._choice(x, w) for w in prototypes])
            order = np.argsort(choices)[::-1]

            assigned = False
            soft = np.zeros(len(prototypes), dtype=float)

            if forced is not None:
                # labeled doc: update its own class prototype, record soft scores for all
                for j in range(len(prototypes)):
                    soft[j] = self._match(x, prototypes[j])
                prototypes[forced] = self._update_proto(prototypes[forced], x)
                counts[forced] += 1
                labels[i] = forced
                assigned = True
            else:
                # record similarity to ALL prototypes (needed for multilabel),
                # then pick the best one (in choice order) that passes the vigilance test.
                for j in range(len(prototypes)):
                    soft[j] = self._match(x, prototypes[j])
                for j in order:
                    if soft[j] >= self.rho:
                        prototypes[j] = self._update_proto(prototypes[j], x)
                        counts[j] += 1
                        labels[i] = j
                        assigned = True
                        break

            if not assigned:
                prototypes.append(x.copy()); counts.append(1)
                self.seed_label_of_proto.append(None)
                labels[i] = len(prototypes) - 1
                soft = np.append(soft, 1.0)

            if soft.sum() <= 0:
                soft = np.zeros(len(prototypes)); soft[labels[i]] = 1.0
            else:
                soft = soft / soft.sum()
            memberships.append(soft)

        self.prototypes = np.array(prototypes)
        self.counts = np.array(counts)
        max_len = max(len(m) for m in memberships)
        M = np.zeros((len(memberships), max_len))
        for i, m in enumerate(memberships):
            M[i, :len(m)] = m
        return labels, M

    @staticmethod
    def multilabel_from_memberships(M, top_k=3, threshold=0.15):
        """
        Hybrid multilabel: for each doc keep up to top_k clusters whose membership
        exceeds `threshold`. Always keeps at least the single best cluster.
        Returns a list (per doc) of (cluster_id, membership) sorted desc.
        """
        out = []
        for row in M:
            order = np.argsort(row)[::-1]
            picks = [(int(j), float(row[j])) for j in order[:top_k] if row[j] >= threshold]
            if not picks:                       # guarantee at least one label
                j = int(order[0]); picks = [(j, float(row[j]))]
            out.append(picks)
        return out


# =========================
# EVALUATION
# =========================
def evaluate_clustering(X, labels, pseudo_labels):
    out = {}
    n_clusters = len(np.unique(labels))
    out["n_clusters"] = int(n_clusters)

    if n_clusters >= 2 and len(X) > n_clusters:
        try:
            out["silhouette_cosine"] = float(silhouette_score(X, labels, metric="cosine"))
        except:
            out["silhouette_cosine"] = np.nan
        try:
            out["calinski_harabasz"] = float(calinski_harabasz_score(X, labels))
        except:
            out["calinski_harabasz"] = np.nan
        try:
            out["davies_bouldin"] = float(davies_bouldin_score(X, labels))
        except:
            out["davies_bouldin"] = np.nan
    else:
        out["silhouette_cosine"] = np.nan
        out["calinski_harabasz"] = np.nan
        out["davies_bouldin"] = np.nan

    # NMI / ARI are computed ONLY when real labels exist (semi-supervised / has labels).
    # In fully-unsupervised mode there is nothing to compare against -> NaN (no crash).
    out["NMI_pseudo"] = np.nan
    out["ARI_pseudo"] = np.nan
    if pseudo_labels is not None and has_any_labels(pd.Series(pseudo_labels)):
        s = pd.Series(pseudo_labels)
        mask = s.notna() & (s.astype(str).str.strip() != "")
        if mask.sum() >= 2 and len(np.unique(labels[mask.values])) >= 1:
            le = LabelEncoder()
            y_true = le.fit_transform(s[mask].astype(str))
            try:
                out["NMI_pseudo"] = float(normalized_mutual_info_score(y_true, labels[mask.values]))
                out["ARI_pseudo"] = float(adjusted_rand_score(y_true, labels[mask.values]))
            except Exception:
                pass
    return out

def rho_sweep_and_select(X, pseudo_labels, out_dir: Path, seed_labels=None, metric="fuzzy"):
    """
    Sweeps rho. SEMI-SUPERVISED if seed_labels is provided (non-null entries seed clusters).
    rho is selected by INTERNAL metrics only (silhouette + calinski + davies),
    guarded against over-clustering. NMI/ARI are computed for REPORTING only.
    """
    ensure_dir(out_dir)

    rows = []
    cache = {}
    for rho in CFG.rho_grid:
        model = SSFuzzyART(rho=rho, alpha=CFG.alpha, beta=CFG.beta, metric=metric)
        labels, memberships = model.fit_predict(X, seed_labels=seed_labels)
        metrics = evaluate_clustering(X, labels, pseudo_labels)
        metrics["rho"] = rho
        rows.append(metrics)
        cache[rho] = (metrics, labels, memberships)

    rho_df = pd.DataFrame(rows)

    # over-clustering guard (unsupervised)
    n_docs = len(X)
    rho_df["avg_cluster_size"] = n_docs / rho_df["n_clusters"].replace(0, np.nan)
    elig = rho_df[(rho_df["n_clusters"] >= 2) &
                  (rho_df["n_clusters"] <= 0.20 * n_docs) &
                  (rho_df["avg_cluster_size"] >= 5.0) &
                  (rho_df["silhouette_cosine"] < 0.999) &
                  (rho_df["davies_bouldin"] > 1e-6)].copy()
    if elig.empty:
        elig = rho_df[(rho_df["n_clusters"] >= 2) & (rho_df["silhouette_cosine"] < 0.999)].copy()
    if elig.empty:
        elig = rho_df[rho_df["n_clusters"] >= 2].copy()
    if elig.empty:
        elig = rho_df.copy()

    def _norm(s, hb=True):
        s = s.astype(float); lo, hi = np.nanmin(s), np.nanmax(s)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
            return pd.Series(np.zeros(len(s)), index=s.index)
        z = (s - lo) / (hi - lo); return z if hb else (1.0 - z)

    elig["internal_score"] = (_norm(elig["silhouette_cosine"], True) +
                              _norm(elig["calinski_harabasz"], True) +
                              _norm(elig["davies_bouldin"], False)) / 3.0
    rho_df = rho_df.merge(elig[["rho", "internal_score"]], on="rho", how="left")
    rho_df.to_csv(out_dir / "rho_sweep_metrics.csv", index=False, encoding="utf-8-sig")

    best_rho = elig.loc[elig["internal_score"].idxmax(), "rho"]
    best, best_labels, best_memberships = cache[best_rho]
    best = best.copy()

    plot_bar(rho_df, "rho", "silhouette_cosine", "Rho Sweep - Silhouette", out_dir / "rho_vs_silhouette.png", rotate=0)
    if rho_df["NMI_pseudo"].notna().any():
        plot_bar(rho_df, "rho", "NMI_pseudo", "Rho Sweep - NMI", out_dir / "rho_vs_nmi.png", rotate=0)
    plot_bar(rho_df.dropna(subset=["internal_score"]), "rho", "internal_score",
             "Rho Sweep - Internal Score", out_dir / "rho_vs_internal_score.png", rotate=0)

    print(f"[rho] selected rho={best_rho} (internal score), n_clusters={best.get('n_clusters')}")
    return best, best_labels, best_memberships, rho_df


# =========================
# REPORTING PER SCENARIO
# =========================
def build_cluster_topic_names(df1, labels, top_n=4):
    """
    Returns dict: cluster_id -> short topic name built from the most distinctive
    keywords/words of that cluster (TF-IDF over text_joined). No labels needed.
    Example: 0 -> "polymer | vinyl | پلیمر | اتیلن"
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    texts = df1["text_joined"].fillna("").astype(str).replace("", " ").tolist()
    tfidf = TfidfVectorizer(max_features=5000, token_pattern=r"(?u)\b\w{3,}\b")
    Xt = tfidf.fit_transform(texts)
    vocab = np.array(tfidf.get_feature_names_out())
    names = {}
    for cid in np.unique(labels):
        mask = (labels == cid)
        mean_tfidf = np.asarray(Xt[mask].mean(axis=0)).ravel()
        top_idx = mean_tfidf.argsort()[::-1][:top_n]
        names[int(cid)] = " | ".join(vocab[top_idx])
    return names


def save_cluster_outputs(df1, X_final, labels, memberships, summary, out_dir: Path):
    ensure_dir(out_dir)

    # keep human-readable columns including titles so a paper can be checked by its number
    keep_cols = [c for c in [CFG.doc_id_col, CFG.title_fa_col, CFG.title_en_col,
                             CFG.year_col, CFG.field_col, CFG.pseudo_col] if c in df1.columns]
    out_df = df1[keep_cols].copy()
    out_df["cluster_id"] = labels
    out_df["max_membership"] = memberships.max(axis=1) if memberships.ndim == 2 else 1.0

    # ---- HYBRID MULTILABEL: up to top_k clusters above threshold (one paper -> several clusters) ----
    ml = SSFuzzyART.multilabel_from_memberships(
        memberships, top_k=CFG.multilabel_top_k, threshold=CFG.multilabel_threshold
    )
    # human-readable TOPIC NAME for each cluster (from its distinctive words)
    topic_names = build_cluster_topic_names(df1, labels, top_n=4)
    # save the cluster -> topic-name table
    pd.DataFrame(
        [{"cluster_id": cid, "topic_name": nm} for cid, nm in sorted(topic_names.items())]
    ).to_csv(out_dir / "cluster_topics.csv", index=False, encoding="utf-8-sig")

    def names_for(picks):
        return " || ".join(f"[{cid}] {topic_names.get(int(cid), '')}" for cid, _ in picks)

    out_df["primary_topic"] = [topic_names.get(int(c), "") for c in labels]
    out_df["multi_labels"] = [";".join(str(cid) for cid, _ in picks) for picks in ml]
    out_df["multi_label_topics"] = [names_for(picks) for picks in ml]
    out_df["multi_label_scores"] = [
        ";".join(f"{cid}:{score:.3f}" for cid, score in picks) for picks in ml
    ]
    out_df["n_labels"] = [len(picks) for picks in ml]
    out_df.to_csv(out_dir / "final_doc_cluster_assignments.csv", index=False, encoding="utf-8-sig")

    # a focused multilabel-only file, easy to inspect by paper number
    ml_df = out_df[[CFG.doc_id_col, CFG.title_fa_col, CFG.title_en_col,
                    "cluster_id", "primary_topic", "multi_labels",
                    "multi_label_topics", "multi_label_scores", "n_labels"]].copy()
    ml_df.rename(columns={CFG.doc_id_col: "paper_no",
                          CFG.title_fa_col: "title_fa",
                          CFG.title_en_col: "title_en",
                          "cluster_id": "primary_cluster"}, inplace=True)
    ml_df.to_csv(out_dir / "doc_multilabels.csv", index=False, encoding="utf-8-sig")

    # quick console summary of multilabel coverage
    multi = sum(1 for p in ml if len(p) > 1)
    print(f"[multilabel] {multi} of {len(ml)} papers got >1 label "
          f"({100.0*multi/max(len(ml),1):.1f}%). topic names saved to cluster_topics.csv")

    np.save(out_dir / "final_doc_cluster_memberships.npy", memberships)

    cluster_sizes = (
        pd.Series(labels).value_counts().sort_index().reset_index()
    )
    cluster_sizes.columns = ["cluster_id", "size"]
    cluster_sizes.to_csv(out_dir / "cluster_size_distribution.csv", index=False, encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(cluster_sizes["cluster_id"].astype(str), cluster_sizes["size"])
    ax.set_title("Cluster Size Distribution")
    ax.set_xlabel("cluster_id")
    ax.set_ylabel("size")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(out_dir / "cluster_size_distribution.png", dpi=CFG.fig_dpi)
    plt.close(fig)

    top_clusters = cluster_sizes.sort_values("size", ascending=False).head(CFG.top_clusters_to_report)
    top_clusters.to_csv(out_dir / "top_clusters_summary.csv", index=False, encoding="utf-8-sig")

    rep_docs = []
    for cid in top_clusters["cluster_id"].tolist():
        sub = out_df[out_df["cluster_id"] == cid].copy()
        if len(sub) == 0:
            continue
        rep = sub.iloc[0]
        rep_docs.append({
            "cluster_id": int(cid),
            "paper_no": rep.get(CFG.doc_id_col, ""),
            "title_fa": rep.get(CFG.title_fa_col, ""),
            "pseudo_label": rep.get(CFG.pseudo_col, ""),
            "field": rep.get(CFG.field_col, "")
        })
    pd.DataFrame(rep_docs).to_csv(out_dir / "representative_docs_top_clusters.csv", index=False, encoding="utf-8-sig")

    if X_final.shape[1] >= 2:
        try:
            pca = PCA(n_components=2, random_state=CFG.random_state)
            Z = pca.fit_transform(X_final)
            fig, ax = plt.subplots(figsize=(7, 6))
            sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=18, cmap="tab20", alpha=0.75)
            ax.set_title("PCA scatter of final features by cluster")
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
            fig.tight_layout()
            fig.savefig(out_dir / "pca_clusters_scatter.png", dpi=CFG.fig_dpi)
            plt.close(fig)
        except:
            pass

    save_json(summary, out_dir / "scenario_summary.json")
    pd.DataFrame([summary]).to_csv(out_dir / "scenario_summary.csv", index=False, encoding="utf-8-sig")


# =========================
# RUN ONE SCENARIO
# =========================
def cluster_quality_report(df1, labels, vectorizer_or_texts, out_dir: Path, top_n_words=10, top_n_clusters=15):
    """
    Builds a human-checkable report of cluster coherence (NO labels needed):
      - cluster_quality.csv : per-cluster size + top distinctive words + intra-cluster
                              cosine cohesion (avg pairwise similarity of its docs).
      - cluster_preview.csv : for the biggest clusters, lists paper numbers + titles so you
                              can eyeball whether the papers really share a topic.
    Higher cohesion (closer to 1) = the docs in that cluster are more similar to each other.
    """
    ensure_dir(out_dir)
    from sklearn.feature_extraction.text import TfidfVectorizer
    texts = df1["text_joined"].fillna("").astype(str).tolist()

    # TF-IDF just for naming clusters by their most distinctive words
    tfidf = TfidfVectorizer(max_features=4000, token_pattern=r"(?u)\b\w{3,}\b")
    Xt = tfidf.fit_transform(texts)
    vocab = np.array(tfidf.get_feature_names_out())

    rows = []
    sizes = pd.Series(labels).value_counts()
    for cid, size in sizes.items():
        mask = (labels == cid)
        sub = Xt[mask]
        # top distinctive words = highest mean tf-idf in the cluster
        mean_tfidf = np.asarray(sub.mean(axis=0)).ravel()
        top_idx = mean_tfidf.argsort()[::-1][:top_n_words]
        top_words = " | ".join(vocab[top_idx])
        # intra-cluster cohesion = average pairwise cosine similarity (sampled if large)
        cohesion = np.nan
        if size >= 2:
            idx = np.where(mask)[0]
            if len(idx) > 40:                      # sample to keep it fast
                idx = np.random.RandomState(0).choice(idx, 40, replace=False)
            sims = cosine_similarity(Xt[idx])
            iu = np.triu_indices(len(idx), k=1)
            cohesion = float(sims[iu].mean()) if len(iu[0]) > 0 else np.nan
        rows.append({"cluster_id": int(cid), "size": int(size),
                     "intra_cohesion_cosine": cohesion, "top_words": top_words})

    qdf = pd.DataFrame(rows).sort_values("size", ascending=False)
    qdf.to_csv(out_dir / "cluster_quality.csv", index=False, encoding="utf-8-sig")

    # preview the biggest clusters with paper numbers + titles for manual checking
    prev = []
    for cid in qdf["cluster_id"].head(top_n_clusters):
        mask = (labels == cid)
        sub = df1[mask]
        for _, r in sub.head(8).iterrows():       # up to 8 papers per cluster
            prev.append({
                "cluster_id": int(cid),
                "paper_no": r.get(CFG.doc_id_col, ""),
                "title_fa": str(r.get(CFG.title_fa_col, ""))[:120],
                "title_en": str(r.get(CFG.title_en_col, ""))[:120],
            })
    pd.DataFrame(prev).to_csv(out_dir / "cluster_preview.csv", index=False, encoding="utf-8-sig")

    # quick console summary
    valid = qdf["intra_cohesion_cosine"].dropna()
    print(f"[quality] clusters={len(qdf)} | mean intra-cohesion={valid.mean():.3f} "
          f"| singletons={(qdf['size']==1).sum()} "
          f"| biggest cluster={int(qdf['size'].max())} docs")
    return qdf


def run_one_scenario(df1: pd.DataFrame, scenario_name: str, scenario_cfg: dict, out_root: Path):
    sc_dir = out_root / scenario_name
    ensure_dir(sc_dir)

    t0 = time.time()

    if scenario_cfg["feature_type"] == "lda":
        feat = build_lda_features(df1, sc_dir / "stage2_features")
        X0 = feat["X_features"]
        extra_metrics = {
            "topic_diversity": feat["meta"]["topic_diversity"],
            "best_k_topics": feat["meta"]["best_k"]
        }
    else:
        feat = build_bert_features(df1, sc_dir / "stage2_features")
        X0 = feat["X_features"]
        extra_metrics = {
            "topic_diversity": np.nan,
            "best_k_topics": np.nan
        }

    # ENTM merges redundant LDA TOPICS. It is only meaningful for the LDA representation.
    # Applying it to dense BERT dimensions is not meaningful and harms the embedding,
    # so ENTM is skipped for BERT regardless of the scenario flag.
    feat = scenario_cfg["feature_type"]
    if scenario_cfg["use_entm"] and feat == "lda":
        X_final, entm_meta = run_entm_general(X0, feat, sc_dir / "stage3_entm")
    else:
        X_final = X0.copy()
        reason = "not_applicable_for_bert" if feat == "bert" else "disabled"
        entm_meta = {"entm_applied": False, "reason": reason,
                     "dim_before": int(X0.shape[1]), "dim_after": int(X0.shape[1])}
        save_json(entm_meta, sc_dir / "stage3_entm" / "entm_meta.json")

    # metric: dense BERT embeddings -> cosine; sparse LDA topic vectors -> fuzzy (classic ART)
    metric = "cosine" if feat == "bert" else "fuzzy"

    # SEMI-SUPERVISED if labels exist: feed them as seeds. Otherwise seeds=None (unsupervised).
    _seeds = df1[CFG.pseudo_col].values if has_any_labels(df1[CFG.pseudo_col]) else None
    best_metrics, best_labels, best_memberships, rho_df = rho_sweep_and_select(
        X_final, df1[CFG.pseudo_col], sc_dir / "stage4_5_cluster_eval",
        seed_labels=_seeds, metric=metric
    )

    runtime_total = time.time() - t0

    summary = {
        "scenario": scenario_name,
        "feature_type": scenario_cfg["feature_type"],
        "use_entm": scenario_cfg["use_entm"],
        "n_docs": int(X_final.shape[0]),
        "feature_dim_final": int(X_final.shape[1]),
        "runtime_total_sec": round(runtime_total, 3),
        **extra_metrics,
        **best_metrics
    }

    save_cluster_outputs(
        df1=df1,
        X_final=X_final,
        labels=best_labels,
        memberships=best_memberships,
        summary=summary,
        out_dir=sc_dir
    )

    # cluster coherence report (paper numbers + titles + top words + intra-cohesion)
    cluster_quality_report(df1, best_labels, None, sc_dir)

    return summary


# =========================
# COMPARISON
# =========================
def compare_scenarios(all_results: List[dict], out_dir: Path):
    ensure_dir(out_dir)

    comp_df = pd.DataFrame(all_results)
    comp_df.to_csv(out_dir / "scenarios_metrics_comparison.csv", index=False, encoding="utf-8-sig")

    rank_df = comp_df.copy()

    high_better = ["silhouette_cosine", "calinski_harabasz", "NMI_pseudo", "ARI_pseudo", "topic_diversity"]
    low_better = ["davies_bouldin", "runtime_total_sec"]

    for c in high_better:
        if c in rank_df.columns:
            x = rank_df[c].astype(float)
            rank_df[c + "_norm"] = (x - x.min()) / (x.max() - x.min() + 1e-12)

    for c in low_better:
        if c in rank_df.columns:
            x = rank_df[c].astype(float)
            rank_df[c + "_norm"] = (x.max() - x) / (x.max() - x.min() + 1e-12)

    norm_cols = [c for c in rank_df.columns if c.endswith("_norm")]
    rank_df["final_score"] = rank_df[norm_cols].mean(axis=1)
    rank_df = rank_df.sort_values("final_score", ascending=False)

    rank_df.to_csv(out_dir / "scenarios_ranked.csv", index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(out_dir / "scenarios_comparison.xlsx", engine="xlsxwriter") as writer:
        comp_df.to_excel(writer, sheet_name="metrics", index=False)
        rank_df.to_excel(writer, sheet_name="ranking", index=False)

    figs_dir = out_dir / "figures"
    ensure_dir(figs_dir)

    metric_list = [
        "silhouette_cosine",
        "calinski_harabasz",
        "davies_bouldin",
        "NMI_pseudo",
        "ARI_pseudo",
        "n_clusters",
        "runtime_total_sec",
        "feature_dim_final"
    ]
    for m in metric_list:
        if m in comp_df.columns:
            plot_bar(comp_df, "scenario", m, f"Scenario Comparison - {m}", figs_dir / f"compare_{m}.png")

    return comp_df, rank_df


# =========================
# MAIN
# =========================
def main():
    out_root = Path(CFG.out_root)
    ensure_dir(out_root)

    print("Loading dataset...")
    df = load_dataset()

    print("Preprocessing...")
    df1 = preprocess_dataset(df, out_root / "common_preprocess")

    all_results = []

    for scenario_name, scenario_cfg in SCENARIOS.items():
        print(f"\nRunning {scenario_name} ...")
        result = run_one_scenario(df1, scenario_name, scenario_cfg, out_root)
        all_results.append(result)

    print("\nComparing scenarios...")
    comp_df, rank_df = compare_scenarios(all_results, out_root / "comparison")

    print("\nCreating final ZIP archive...")
    zip_all_outputs(out_root, out_root / "archives" / "all_outputs.zip")

    print("\nDONE.")
    print(f"All outputs saved to: {CFG.out_root}")

main()
